In [ ]:
import os
import requests
import json

URL = "http://domino-irsa-lite-admin.domino-field.svc.cluster.local:8000"

# Configuration - update these for your environment
AWS_ACCOUNT_ID = "946429944765"
AWS_PARTITION = "aws"  # or "aws-us-gov"
DEFAULT_ROLE = f"arn:{AWS_PARTITION}:iam::{AWS_ACCOUNT_ID}:role/acme_domino_default_role"

def get_domino_token():
    return requests.get(f"{os.environ['DOMINO_API_PROXY']}/access-token").text

def auth_headers():
    return {"Authorization": f"Bearer {get_domino_token()}"}

# Health check
resp = requests.get(f"{URL}/health/healthz")
print(f"Status: {resp.status_code}")
print(resp.json())

## List K8s Service Accounts

Lists all service accounts in the compute namespace with the `domino/user-sa=true` label.

In [ ]:
resp = requests.get(f"{URL}/k8s/serviceaccounts", headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Sync Domino Users to K8s Service Accounts

Creates/updates/deletes K8s service accounts to match Domino users. Each SA gets the default AWS role annotation.

In [ ]:
# Dry run first to see what would change
body = {
    "default_aws_role_arn": DEFAULT_ROLE,
    "dry_run": True
}
resp = requests.post(f"{URL}/k8s/users/sync", json=body, headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Apply User Sync (dry_run=False)

Actually apply the changes after reviewing the dry run output.

In [ ]:
# Apply the changes (set dry_run=False)
body = {
    "default_aws_role_arn": DEFAULT_ROLE,
    "dry_run": False
}
resp = requests.post(f"{URL}/k8s/users/sync", json=body, headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## List Keycloak IRSA Groups

Lists all groups under the IRSA root group with their members.

In [ ]:
resp = requests.get(f"{URL}/keycloak/groups", headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Get IRSA Trust Policies

Shows the current trust policy subs for each IAM role mapped to a Keycloak group.

In [ ]:
resp = requests.get(f"{URL}/iam/irsa/trust", headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Sync IRSA Trust Policies (Dry Run)

Syncs IAM role trust policies to match Keycloak group memberships. Run with dry_run=True first.

In [ ]:
body = {"dry_run": True}
resp = requests.post(f"{URL}/iam/irsa/sync", json=body, headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Apply IRSA Sync (dry_run=False)

Actually apply the trust policy changes after reviewing the dry run output.

In [ ]:
body = {"dry_run": False}
resp = requests.post(f"{URL}/iam/irsa/sync", json=body, headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Add User to Keycloak Group

Adds a user to an IRSA role group. The group_name can be just the role ARN or the full path.

In [ ]:
body = {
    "username": "integration-test",
    "group_name": f"arn:{AWS_PARTITION}:iam::{AWS_ACCOUNT_ID}:role/acme_domino_role_1"
}
resp = requests.post(f"{URL}/keycloak/groups/member", json=body, headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))

## Remove User from Keycloak Group

Removes a user from an IRSA role group.

In [ ]:
body = {
    "username": "integration-test",
    "group_name": f"arn:{AWS_PARTITION}:iam::{AWS_ACCOUNT_ID}:role/acme_domino_role_1"
}
resp = requests.delete(f"{URL}/keycloak/groups/member", json=body, headers=auth_headers())
print(f"Status: {resp.status_code}")
print(json.dumps(resp.json(), indent=2))